In [34]:
import matplotlib.pyplot as plt
import sympy as sp
%matplotlib inline
from sympy import integrate, init_printing

# Transformada de Laplace

Si $f(t)$ es una función definida en $[0,\infty[$ con $t$ y $f$ reales, entonces la transformada de Laplace de la función $f$ se denota por $\mathcal{L}\lbrace f(t) \rbrace = F(s)$ y se define como la integral
$$ \mathcal{L}\lbrace f(t) \rbrace = F(s) = \int_{0}^{\infty} f(t) e^{-st} dt $$
siempre que la anterior sea convergente.

## Propiedades de las transformadas de Laplace
Las Transformadas de Laplace poseen algunas propiedades que también nos van a facilitar el trabajo de resolverlas, algunas de ellas son:

* La Transformada de Laplace es un operador lineal:

Esta propiedad nos dice la Transformada de Laplace de una suma, es igual a la suma de las Transformadas de Laplace de cada uno de los términos. Es decir:

$$\mathcal{L}\lbrace c_1f_1(t)+c_2f_2(t)\rbrace=c_1\mathcal{L}\lbrace f_1(t)\rbrace+c_2\mathcal{L}\lbrace f_2(t)\rbrace.$$

* La Transformada de Laplace de la primera derivada: 

Esta propiedad nos dice que si $f(t)$ es continua y $f'(t)$ es continua en el intervalo $ 0 \leq t \leq \alpha $. Entonces la Transformada de Laplace de la primera derivada es:
$$\mathcal{L}\lbrace f'(t)\rbrace =s\mathcal{L}\lbrace f(t)\rbrace−f(0).$$

* La Transformada de Laplace de derivadas de orden superior: 

Esta propiedad es la generalización de la propiedad anterior para derivadas de orden $n$. Su formula es:
$$\mathcal{L}\lbrace f^{(n)}(t)\lbrace=s^n\mathcal{L}\lbrace f(t)\lbrace −s^{n−1}f(0)-\cdots−f^{(n−1)}(0)= s^n\mathcal{L}\lbrace f(t)\lbrace −\sum_{i=1}^n s^{n−i}f^{(i−1)}(0)$$

## Resolución de ecuaciones diferenciales con transformada de Laplace
La principal ventaja de utilizar Transformadas de Laplace es que cambia la Ecuación diferencial en una ecuación algebraica, lo que simplifica el proceso para calcular su solución. La única parte complicada es encontrar las transformaciones y las inversas de las transformaciones de los varios términos de la Ecuación diferencial que queramos resolver. Veamos entonces como `Python` y `SymPy` nos ayudan a resolver Ecuaciones diferenciales utilizando Transformadas de Laplace. 


### Ejemplo 01

Resuelva la siguiente ecuación:

$$ y''+3y'+2y=0$$ 

con las condiciones iniciales: $y(0)=2$ y $y'(0)=−3$.


In [35]:
import sympy as sp
import matplotlib.pyplot as plt
#from sympy import integrate, init_printing #importamos paquete para integrar


# Ejemplo de transformada de Laplace
# Defino las incognitas
t = sp.symbols("t", positive=True)
y = sp.Function("y")

# Defino la ecuación
edo = y(t).diff(t, t) + 3*y(t).diff(t) + 2*y(t)
sp.Eq(edo, 0)

Eq(2*y(t) + 3*Derivative(y(t), t) + Derivative(y(t), (t, 2)), 0)

In [47]:
# simbolos adicionales.
s, Y = sp.symbols("s, Y", real=True)

# Calculo la transformada de Laplace 
L_edo = sp.laplace_transform(edo, t, s, noconds=True)
sp.Eq(L_edo)

TypeError: Equality.__new__() missing 1 required positional argument: 'rhs'

En este resultado, al aplicar la función `sp.laplace_transform` sobre la derivada de $y(t)$, `SymPy` nos devuelve un termino con la forma 
$$\mathcal{L}\lbrace \frac{d}{dt}y(t)\rbrace (s)$$

Esto es una complicación si queremos arribar a una ecuación algebraica. Por tanto antes de proceder, debemos utilizar la siguiente función para resolver este inconveniente.

In [37]:
def laplace_transform_derivatives(e):
    """
    Evalua las transformadas de Laplace de derivadas de funciones sin evaluar.
    """
    if isinstance(e, sp.LaplaceTransform):
        if isinstance(e.args[0], sp.Derivative):
            d, t, s = e.args 
            n = len(d.args) - 1
            return ((s**n) * sp.LaplaceTransform(d.args[0], t, s) - sum([s**(n-i) * sp.diff(d.args[0], t, i-1).subs(t, 0) for i in range(1, n+1)]))
        
    if isinstance(e, (sp.Add, sp.Mul)):
        t = type(e) 
        return t(*[laplace_transform_derivatives(arg) for arg in e.args])
    
    return e

In [38]:
# Aplicamos la nueva funcion para evaluar las transformadas de Laplace
# de derivadas
L_edo_2 = laplace_transform_derivatives(L_edo)
sp.Eq(L_edo_2, 0)

Eq(s**2*LaplaceTransform(y(t), t, s) + 3*s*LaplaceTransform(y(t), t, s) - s*y(0) + 2*LaplaceTransform(y(t), t, s) - 3*y(0) - Subs(Derivative(y(t), t), t, 0), 0)

In [46]:
# reemplazamos la transfomada de Laplace de y(t) por la incognita Y
# para facilitar la lectura de la ecuación.
L_edo_3 = L_edo_2.subs(sp.laplace_transform(y(t), t, s), Y)
sp.Eq(L_edo_3, 0)

Eq(s**2*LaplaceTransform(y(t), t, s) + 3*s*LaplaceTransform(y(t), t, s) - s*y(0) + 2*LaplaceTransform(y(t), t, s) - 3*y(0) - Subs(Derivative(y(t), t), t, 0), 0)

In [40]:
# Definimos las condiciones iniciales
ics = {y(0): 2, y(t).diff(t).subs(t, 0): -3}
ics

{y(0): 2, Subs(Derivative(y(t), t), t, 0): -3}

In [41]:
# Aplicamos las condiciones iniciales
L_edo_4 = L_edo_3.subs(ics)
L_edo_4

s**2*LaplaceTransform(y(t), t, s) + 3*s*LaplaceTransform(y(t), t, s) - 2*s + 2*LaplaceTransform(y(t), t, s) - 3

In [42]:
# Resolvemos la ecuación y arribamos a la Transformada de Laplace
# que es equivalente a nuestra ecuación diferencial
Y_sol = sp.solve(L_edo_4, Y)
Y_sol

[]

In [43]:
# Por último, calculamos al inversa de la Transformada de Laplace que 
# obtuvimos arriba, para obtener la solución de nuestra ecuación diferencial.
y_sol = sp.inverse_laplace_transform(Y_sol[0], s, t)
y_sol

IndexError: list index out of range

### Ejercicio 01

Usando transformada de la Laplace, resuelva:
$$ y''+y+1=e^{t}\sin(2t)+\cos(2t-1) \quad ; y(0)=1 ; y'(0)=2$$

### Ejercicio 02

Usando transformada de la Laplace, resuelva:
$$ y''+y=\frac{1}{\sqrt{t}} + \sqrt{t}  \quad ; y(0)=1 ; y'(0)=2$$

In [ ]:
t,s = sp.symbols("t s", real = True, positive = True)
sp.laplace_transform(sp.sqrt(1/t),t,s,noconds = True)

sqrt(pi)/sqrt(s)

In [ ]:
sp.laplace_transform(sp.sqrt(t),t,s,noconds = True)

sqrt(pi)/(2*s**(3/2))

In [ ]:
F = (s+2)/(s**2 + 1) + sp.sqrt(sp.pi)*(2*s+1)/(2*sp.sqrt(s)*s*(s**2 + 1)) 
F



(s + 2)/(s**2 + 1) + sqrt(pi)*(2*s + 1)/(2*s**(3/2)*(s**2 + 1))

In [ ]:
sp.inverse_laplace_transform(F,s,t,noconds=True)

sqrt(t) + sqrt(2)*sqrt(pi)*(sin(t)*fresnelc(sqrt(2)*sqrt(t)/sqrt(pi)) - sin(t)*fresnels(sqrt(2)*sqrt(t)/sqrt(pi))/2 - cos(t)*fresnelc(sqrt(2)*sqrt(t)/sqrt(pi))/2 - cos(t)*fresnels(sqrt(2)*sqrt(t)/sqrt(pi))) + 2*sin(t) + cos(t)

In [ ]:
sp.__version__

'1.11.1'

### Ejercicio 03

Usando transformada de la Laplace, resuelva:
$$ ty''+y'+y = 0$$

sqrt(pi)*(InverseLaplaceTransform(1/(s**(7/2) + s**(3/2)), s, t, _None) + 2*Derivative(InverseLaplaceTransform(1/(s**(7/2) + s**(3/2)), s, t, _None), t))/2 + 2*sin(t) + cos(t)

### Ejercicio 04:

* a) Con la ayuda de la librería `sympy`, determine $\mathcal{L}\left\lbrace \dfrac{1}{\sqrt{t}} \right\rbrace$. A partir de su resultado deduzca $\mathcal{L}^{-1}\left\lbrace \dfrac{1}{\sqrt{s}} \right\rbrace$ 

* b) Usando el resultado anterior, resuelva la ecuación integral

$$ C = \int_{0}^{t} \frac{f(t)}{\sqrt{t-\tau}} d\tau $$

para $C\in \mathbb{R}$ y $f$ función incógnita.

Recuerde que
$$(f * g)(t) =  \int_{0}^{t} f(t)g(t-\tau) d\tau $$
y
$$ \mathcal{L}\lbrace f * g \rbrace = \mathcal{L}\lbrace f \rbrace \mathcal{L}\lbrace g \rbrace $$


### Ejercicio 05

La función error de aparece en teoría de probabilidad y se define como
$$erf(t)=\frac{2}{\sqrt{\pi}}\int_{0}^{t}e^{-x^2}dx$$

esta función está incorporada en la librería `sympy` definida por `sp.erf(t)` (definiendo previamente el simbolo `t`)

* a) Usando el comando `checkodesol` verifique que $y=c_1 erf(t)+c_2$ es solución general de la EDO homogénea
$$y'' + 2t y'=0$$

* b) Calcule $\mathcal{L}\lbrace erf(\sqrt{t}) \rbrace$.

* c) Sin utilizar el comando `dsol`, pero con ayuda de `sympy`, resuelva el PVI

$$y' = erf(t) + t \quad , \quad y(0)=-1$$

### Ejercicio 06:

Se sabe que, para $f(t)=t^n$ con $n\in \mathbb{N}$
$$\mathcal{L}\lbrace t^n \rbrace = \frac{n!}{s^{n+1}}$$
sin embargo, esta fórmula deja de ser cierta si $n \in \mathbb{R}$.

* a) Usando librería sympy, calcule $\mathcal{L}\lbrace t^{\frac{p}{2}} \rbrace$ para $p=1,3,5,7$. 

Sugerencia: Se recomienda utilizar `sp.Rational(a,b)` para representar números racionales de la forma $\frac{a}{b}$.

* b) A partir de los resultados de a) deduzca una fórmula para $\mathcal{L}\lbrace t^{\frac{2n-1}{2}} \rbrace$ para $ n \in \mathbb{N}$

* c) Usando estos resultados, resuelva el PVI

$$y' = \sqrt{t} \quad ; \quad y(0)=1$$

In [ ]:
# a) Calcular la transformada de Laplace de t^(p/2) para p = 1, 3, 5, 7
p_values = [1, 3, 5, 7, 9]
laplace_transforms = [sp.laplace_transform(t**sp.Rational(p, 2), t, s, noconds=True) for p in p_values]
laplace_transforms



[sqrt(pi)/(2*s**(3/2)),
 3*sqrt(pi)/(4*s**(5/2)),
 15*sqrt(pi)/(8*s**(7/2)),
 105*sqrt(pi)/(16*s**(9/2)),
 945*sqrt(pi)/(32*s**(11/2))]

In [ ]:
from sympy import gamma

# Definir n como un símbolo
n = sp.symbols('n', integer=True, positive=True)

# Calcular la fórmula general para la transformada de Laplace
laplace_general = gamma((2*n + 1)/2) / s**((2*n + 1)/2)
laplace_general

s**(-n - 1/2)*gamma(n + 1/2)